In [1]:
%useLatestDescriptors

%use dataframe
%use kandy
%use ktor-client

In [2]:
val serviceKeyFilePath = "/Users/unchil/AndroidStudioProjects/OceanWaterInfo/collectionServer/src/main/resources/application.json"
val configData = DataRow.readJson(path=serviceKeyFilePath)

In [3]:
import java.time.LocalDateTime
import java.time.format.DateTimeFormatter

val formatter = DateTimeFormatter.ofPattern("yyyy-MM-dd HH:mm:ss")
val now = LocalDateTime.now()
val previousHour = now.minusHours(24)

val wtch_dt_start = previousHour.format(formatter)
val wtch_dt_end = now.format(formatter)

println("wtch_dt_start:${wtch_dt_start}, wtch_dt_end:${wtch_dt_end}")

wtch_dt_start:2026-07-20 17:56:27, wtch_dt_end:2026-07-21 17:56:27


In [5]:
import java.net.URLEncoder
import java.nio.charset.StandardCharsets

val url = "${configData.MOF_API.endPoint}/${configData.MOF_API.subPath}" +
        "?wtch_dt_start=${URLEncoder.encode(wtch_dt_start, StandardCharsets.UTF_8.toString())}" +
        "&wtch_dt_end=${URLEncoder.encode(wtch_dt_end, StandardCharsets.UTF_8.toString())}" +
        "&numOfRows=1000" +
        "&ServiceKey=${configData.MOF_API.apikey}"

In [6]:
@file:DependsOn("org.json:json:20250107")

In [7]:
import org.json.XML

fun loadData(path:String):DataFrame<*> {
    var requestPage = 1
    val rows = mutableListOf<DataFrame<*>>()

    try {
        do{
            val pagePath = "$path&pageNo=$requestPage"
            val response = http.get(pagePath)
            if (response.status.value == 200) {
                try {
                    XML.toJSONObject(response.bodyAsText()).let { jsonData ->
                        val df = DataFrame.readJson(jsonData.toString().byteInputStream())
                        val result = df.get("response").get("body").get("items").get("item")[0] as DataFrame<*>
                        requestPage += 1
                        rows.add(result)
                    }
                } catch(e: Exception) {
                    println(e.localizedMessage)
                    break
                }
            } else {
                println("${response.status.description}")
            }
        } while (requestPage < 500 )
    } catch(e:Exception ){
        println(e.localizedMessage)
    }
    return rows.concat()
}


In [8]:
val dfRaw = loadData(url)

dfRaw.describe()

Column not found: 'items'


name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2128,1080,0,0.260000,61,5.088249,2.295914,0.250000,3.882083,5.481000,6.390000,18.955999
rtmWqChpla,Comparable<*>,2128,1310,0,,177,null,null,null,null,null,null,null
rtmWqBgalgsQy,String,2128,1,0,,2128,null,null,,,,,
rtmWqWtchStaCd,String,2128,14,0,SEA6001,177,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Int,2128,2128,0,1,1,1064.500000,614.445007,1,532.416667,1064.500000,1596.583333,2128
rtmWqTu,Int,2128,159,0,5,194,25.368421,33.498284,0,5.000000,11.000000,32.583333,232
ph,Double,2128,137,0,7.500000,52,7.671631,0.283060,7.020000,7.480000,7.635000,7.920000,8.960000
rtmWqSlnty,Number,2128,1973,0,32.705002,4,21.822985,10.287468,0.020000,13.976000,26.645000,29.538000,34.032001
rtmWqCndctv,Float,2128,2048,0,44.272999,3,34.075283,15.398833,0.046000,23.302416,39.775000,45.050584,54.451000
rtmWqWtchDtlDt,String,2128,189,0,2026-07-20 18:10:00.0,14,null,null,2026-07-20 18:00:00.0,2026-07-20 23:55:00.0,2026-07-21 05:45:00.0,2026-07-21 11:40:00.0,2026-07-21 17:30:00.0


In [19]:
val df = dfRaw.parse().remove{ rtmWqBgalgsQy }.convert { rtmWqChpla }.with {
    val value = it.toString().trim()
    if (value.isNullOrBlank()) 0.0 else value.toDouble()
}.convert {  colsOf<Number>() }.with { it.toString().trim().toDouble()}


df.describe()

name,type,count,unique,nulls,top,freq,mean,std,min,p25,median,p75,max
rtmWqDoxn,Double,2128,1080,0,0.260000,61,5.088249,2.295914,0.250000,3.882083,5.481000,6.390000,18.955999
rtmWqChpla,Double,2128,1310,0,0.000000,177,5.212666,5.245419,0.000000,1.300000,3.014000,7.920000,32.040000
rtmWqWtchStaCd,String,2128,14,0,SEA6001,177,null,null,NEP1002,NEP3001,SEA2005,SEA5002,SEA7002
num,Double,2128,2128,0,1.000000,1,1064.500000,614.445007,1.000000,532.416667,1064.500000,1596.583333,2128.000000
rtmWqTu,Double,2128,159,0,5.000000,194,25.368421,33.498284,0.000000,5.000000,11.000000,32.583333,232.000000
ph,Double,2128,137,0,7.500000,52,7.671631,0.283060,7.020000,7.480000,7.635000,7.920000,8.960000
rtmWqSlnty,Double,2128,1973,0,32.705002,4,21.822985,10.287468,0.020000,13.977667,26.647000,29.541500,34.032001
rtmWqCndctv,Double,2128,2048,0,44.273000,3,34.075283,15.398833,0.046000,23.302417,39.775000,45.050583,54.451000
rtmWqWtchDtlDt,LocalDateTime,2128,189,0,2026-07-20T18:10,14,null,null,2026-07-20T18:00,2026-07-20T23:55,2026-07-21T05:45,2026-07-21T11:40,2026-07-21T17:30
rtmWtchWtem,Double,2128,750,0,26.990000,13,26.373863,2.334184,19.740000,24.990000,26.450001,28.180000,30.730000


In [35]:
val newColunmNames = listOf( "용존산소", "클로로필", "관측정점코드", "순번", "탁도", "수소이온농도", "염분", "전기전도도", "일시", "수온")
val renamePairs = df.columnNames().zip(newColunmNames).toTypedArray()
val renamedDf = df.rename(*renamePairs)
renamedDf.columnNames()

[용존산소, 클로로필, 관측정점코드, 순번, 탁도, 수소이온농도, 염분, 전기전도도, 일시, 수온]

In [36]:
val removedDf = renamedDf.move { 순번 and 일시 }.toStart()

removedDf.head(5)

순번,일시,용존산소,클로로필,관측정점코드,탁도,수소이온농도,염분,전기전도도,수온
1.000000,2026-07-20T18:00,5.412000,2.554000,NEP2001,7.000000,7.750000,15.994000,26.180000,26.120001
2.000000,2026-07-20T18:00,6.580000,14.390000,SEA6001,45.000000,7.960000,29.934999,46.210000,26.629999
3.000000,2026-07-20T18:00,6.550000,10.790000,SEA7002,7.000000,7.560000,26.754000,39.754000,22.580000
4.000000,2026-07-20T18:00,2.580000,1.854000,SEA2007,33.000000,7.820000,32.443001,52.315000,23.900000
5.000000,2026-07-20T18:00,1.300000,0.000000,SEA1005,14.000000,7.320000,0.779000,1.561000,28.870001


In [37]:
removedDf
    .select{  일시 and 수온 and 관측정점코드   }
    .plot{
        layout {
            title = "한국 해수 정보"
            size = 2600 to 600
        }
        x(일시) { axis.name = "관측일시"}
        y(수온) {axis.name ="수온 °C"}
        line{
            color(관측정점코드){
                //   scale = continuous(Color.GREEN..Color.RED)
                legend{
                    name = "관측정점코드"
                }
            }
        }
    }

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.8.2/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="OdOYQT" ></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 2600.0, 
 height: 600.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("OdOYQT");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"ggtitle":{
"text":"한국 해수 정보"
},
"mapping":{
},
"data":{
"일시":[1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.7845704E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.784571E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845713E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845719E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845722E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845728E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845731E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.7845737E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.784574E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845746E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845749E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845755E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845758E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845764E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845767E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845773E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845776E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845782E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845785E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845791E12,1.7845794E12,1.7845794E12,1